# AI Job Market Intelligence Platform

## 4. Feature Engineering

### Objective

The objective of this notebook is to create meaningful analytical features from the cleaned AI job market dataset.

These features will be used for:

- Exploratory Data Analysis
- SQL Analysis
- Business Insights
- Power BI Dashboard

### Feature Engineering Activities

1. Experience grouping
2. Salary segmentation
3. Seniority classification
4. Remote work classification
5. Salary range calculation
6. Job posting time features
7. Skill indicators
8. Feature validation

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("../data/processed/ai_jobs_cleaned.csv")

In [3]:
df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k)
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k)
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k)
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k)
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k)


In [4]:
df.shape

(1500, 25)

In [5]:
df.columns

Index(['job_id', 'job_title', 'job_category', 'experience_level',
       'years_of_experience', 'education_required', 'annual_salary_usd',
       'salary_min_usd', 'salary_max_usd', 'city', 'country', 'remote_work',
       'company_size', 'industry', 'required_skills', 'ai_salary_premium_pct',
       'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10',
       'posting_year', 'posting_month', 'is_senior', 'is_remote_friendly',
       'is_llm_role', 'salary_tier'],
      dtype='object')

In [6]:
# copy dataset
feature_df = df.copy()

In [7]:
# Experience group 
def experience_group(years):
    if pd.isna(years):
        return "Unknown"
    elif years <= 2:
        return "Entry Level"
    elif years <= 5:
        return "Mid Level"
    elif years <= 10:
        return "Senior Level"
    else:
        return "Expert Level"
    
feature_df["experience_group"] = (
    feature_df["years_of_experience"]
    .apply(experience_group)
)

In [8]:
feature_df["experience_group"].value_counts() 

experience_group
Senior Level    797
Mid Level       510
Entry Level     111
Expert Level     82
Name: count, dtype: int64

In [9]:
# salary band

def salary_band(salary):
    if pd.isna(salary):
        return "Unknown"
    elif salary < 60000:
        return "Low"
    elif salary < 100000:
        return "Medium"
    elif salary < 150000:
        return "High"
    else:
        return "Very High"

feature_df["salary_band"] = (
    feature_df["annual_salary_usd"]
    .apply(salary_band)
)

In [10]:
feature_df["salary_band"].value_counts()

salary_band
Very High    1062
High          400
Medium         38
Name: count, dtype: int64

In [11]:
# salary range
feature_df["salary_range_usd"] = (
    feature_df["salary_max_usd"] - 
    feature_df["salary_min_usd"]
)
feature_df.head(5)

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000


In [12]:
# remote work flag
feature_df["is_fully_remote"] = (
    feature_df["remote_work"]
    .str.lower()
    .eq("remote")
)
feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False


In [13]:
# seniority classification 
feature_df["job_title"].value_counts()

job_title
LLM Engineer                75
Robotics Engineer (AI)      74
Prompt Engineer             71
Generative AI Engineer      71
AI Product Manager          70
Multimodal AI Engineer      67
Senior Data Scientist       66
AI Compliance Manager       66
Senior ML Engineer          64
AI Engineer                 64
AI Business Analyst         62
Data Scientist              61
Deep Learning Engineer      58
AI Agent Developer          57
AI Ethics Officer           56
ML Engineer                 55
NLP Engineer                55
AI Infrastructure Eng       55
RAG Engineer                53
AI Solutions Architect      52
Data Engineer (AI)          51
MLOps Engineer              51
AI Security Engineer        50
AI Research Scientist       50
Computer Vision Engineer    46
Name: count, dtype: int64

In [14]:
# passing year / quarter
month_number = pd.to_datetime(
    feature_df["posting_month"],
    format="%B",
    errors="coerce"
).dt.month

In [15]:
feature_df["posting_quarter"] = (
    "Q" + month_number.astype("Int64").astype(str)
)

In [16]:
feature_df["posting_month"].unique()

array([ 3,  1,  9,  8,  2, 11, 10,  7, 12,  5,  6,  4])

In [17]:
# AI/LLM Job Flags
feature_df["is_llm_role"].value_counts()

is_llm_role
0    1173
1     327
Name: count, dtype: int64

In [18]:
# python required
feature_df["requires_python"] = (
    feature_df["required_skills"]
    .fillna("")
    .str.lower()
    .str.contains("python")
)
feature_df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False


In [19]:
# SQL required
feature_df["requires_sql"] = (
    feature_df["required_skills"]
    .fillna("")
    .str.lower()
    .str.contains("sql")
)

feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False


In [20]:
# machine Learning required
feature_df["requires_machine_learning"] = (
    feature_df["required_skills"]
    .fillna("")
    .str.lower()
    .str.contains("machine learning")
)

feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False,False
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True,False
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False,False
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False,False


In [21]:
# Deep Learning
feature_df["requires_deep_learning"] = (
    feature_df["required_skills"]
    .fillna("")
    .str.lower()
    .str.contains("deep learning")
)
feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False,False
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False,False,False
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True,False,False
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False,False,False
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False,False,False


In [22]:
# Generative AI
feature_df["requires_generative_ai"] = (
    feature_df["required_skills"]
    .fillna("")
    .str.lower()
    .str.contains("generative ai|genai")
)
feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False,False,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False,False,False
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False,False,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False,False,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False,False,False,False
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True,False,False,False
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False,False,False,False
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False,False,False,False


In [23]:
#  Salary Per Experience
feature_df["salary_per_experience"] = np.where(
    feature_df["years_of_experience"] > 0,
    feature_df["annual_salary_usd"] /
    feature_df["years_of_experience"],
    np.nan
)
feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai,salary_per_experience
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False,False,False,34142.857143
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False,False,False,83000.000000
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False,False,False,90000.000000
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False,False,False,53666.666667
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False,False,False,56600.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False,False,False,False,32750.000000
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True,False,False,False,18111.111111
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False,False,False,False,12700.000000
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False,False,False,False,36250.000000


In [24]:
# Remote-Friendly Flag 
feature_df["is_remote_work"] = (
    feature_df["remote_work"]
    .str.lower()
    .isin(["remote", "hybrid"])
)
feature_df

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai,salary_per_experience,is_remote_work
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False,False,False,34142.857143,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False,False,False,83000.000000,True
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False,False,False,90000.000000,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False,False,False,53666.666667,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False,False,False,56600.000000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,AIJOB1496,Multimodal AI Engineer,AI Engineering,Lead (10+ yrs),8.0,Master's,262000.0,160000,300000,Toronto,Canada,Hybrid,SME (51-500),Retail,Python|Vision-Language Models|Fine-tuning|PyTo...,8.8,NaN,23.6,7.5,2026,1,1,1,0,Senior ($200-300k),Senior Level,Very High,140000,False,Q<NA>,True,False,False,False,False,32750.000000,True
1496,AIJOB1497,NLP Engineer,AI Engineering,Senior (6-9 yrs),9.0,Bootcamp/Self-taught,163000.0,145000,290000,Berlin,Germany,On-Site,Mid-size (501-5000),Manufacturing,Transformers|Text Processing|LLMs|Hugging Face...,6.7,NaN,66.5,7.8,2025,3,1,0,0,Upper-Mid ($150-200k),Senior Level,Very High,145000,False,Q<NA>,False,True,False,False,False,18111.111111,False
1497,AIJOB1498,AI Compliance Manager,Governance,Mid (3-5 yrs),10.0,Master's,127000.0,100000,200000,Zurich,Switzerland,Hybrid,SME (51-500),Manufacturing,Documentation|Risk Management|Legal Knowledge|...,10.9,NaN,15.3,8.3,2026,3,0,1,0,Mid ($100-150k),Senior Level,High,100000,False,Q<NA>,False,False,False,False,False,12700.000000,True
1498,AIJOB1499,NLP Engineer,AI Engineering,Lead (10+ yrs),4.0,Bachelor's,145000.0,145000,290000,Bangalore,India,Hybrid,Enterprise (5000+),Media,LLMs|Hugging Face|Python|Transformers|BERT|Tex...,8.5,NaN,78.8,7.4,2025,5,1,1,0,Mid ($100-150k),Mid Level,High,145000,False,Q<NA>,True,False,False,False,False,36250.000000,True


In [25]:
feature_df.columns

Index(['job_id', 'job_title', 'job_category', 'experience_level',
       'years_of_experience', 'education_required', 'annual_salary_usd',
       'salary_min_usd', 'salary_max_usd', 'city', 'country', 'remote_work',
       'company_size', 'industry', 'required_skills', 'ai_salary_premium_pct',
       'demand_score', 'demand_growth_yoy_pct', 'benefits_score_10',
       'posting_year', 'posting_month', 'is_senior', 'is_remote_friendly',
       'is_llm_role', 'salary_tier', 'experience_group', 'salary_band',
       'salary_range_usd', 'is_fully_remote', 'posting_quarter',
       'requires_python', 'requires_sql', 'requires_machine_learning',
       'requires_deep_learning', 'requires_generative_ai',
       'salary_per_experience', 'is_remote_work'],
      dtype='object')

In [26]:
feature_df.head()

,job_id,job_title,job_category,experience_level,years_of_experience,education_required,annual_salary_usd,salary_min_usd,salary_max_usd,city,country,remote_work,company_size,industry,required_skills,ai_salary_premium_pct,demand_score,demand_growth_yoy_pct,benefits_score_10,posting_year,posting_month,is_senior,is_remote_friendly,is_llm_role,salary_tier,experience_group,salary_band,salary_range_usd,is_fully_remote,posting_quarter,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai,salary_per_experience,is_remote_work
0,AIJOB0001,AI Agent Developer,AI Engineering,Senior (6-9 yrs),7.0,Master's,239000.0,155000,290000,Boston,Usa,On-Site,Startup (1-50),Finance,APIs|Planning Systems|Python|Cloud|SQL|Leadership,13.1,NaN,16.9,6.8,2026,3,1,0,1,Senior ($200-300k),Senior Level,Very High,135000,False,Q<NA>,True,True,False,False,False,34142.857143,False
1,AIJOB0002,Prompt Engineer,AI Engineering,Senior (6-9 yrs),2.0,Bachelor's,166000.0,90000,200000,London,Uk,Hybrid,Enterprise (5000+),Finance,Python|Documentation|LLM APIs|Prompt Design|NL...,5.4,NaN,11.6,6.2,2026,1,1,1,1,Upper-Mid ($150-200k),Entry Level,Very High,110000,False,Q<NA>,True,False,False,False,False,83000.000000,True
2,AIJOB0003,LLM Engineer,AI Engineering,Senior (6-9 yrs),4.0,Associate's,360000.0,160000,300000,Seattle,Usa,Fully Remote,Big Tech (FAANG+),Finance,Vector DBs|Python|Prompt Engineering|Fine-tuni...,9.1,NaN,42.7,7.7,2026,1,1,1,1,Elite (>$300k),Mid Level,Very High,140000,False,Q<NA>,True,False,False,False,False,90000.000000,False
3,AIJOB0004,Data Engineer (AI),Data Engineering,Senior (6-9 yrs),3.0,Bachelor's,161000.0,130000,220000,Singapore,Singapore,Fully Remote,SME (51-500),Technology,Feature Stores|Spark|ETL|Airflow|dbt|SQL|Pytho...,12.0,NaN,6.7,9.5,2026,3,1,1,0,Upper-Mid ($150-200k),Mid Level,Very High,90000,False,Q<NA>,True,True,False,False,False,53666.666667,False
4,AIJOB0005,AI Product Manager,Product,Lead (10+ yrs),5.0,Bootcamp/Self-taught,283000.0,140000,260000,Los Angeles,Usa,Fully Remote,Enterprise (5000+),Automotive,Data Analysis|Stakeholder Mgmt|Agile|Cloud|Pro...,9.4,NaN,17.3,8.9,2026,1,1,1,0,Senior ($200-300k),Mid Level,Very High,120000,False,Q<NA>,False,False,False,False,False,56600.000000,False


In [27]:
## Feature Validation 
# Experience Groups 
feature_df[
    ["years_of_experience", "experience_group"]
].head(20)

,years_of_experience,experience_group
0,7.0,Senior Level
1,2.0,Entry Level
2,4.0,Mid Level
3,3.0,Mid Level
4,5.0,Mid Level
5,5.0,Mid Level
6,5.0,Mid Level
7,7.0,Senior Level
8,3.0,Mid Level
9,4.0,Mid Level


In [28]:
# Salary 
feature_df[
    [
        "annual_salary_usd",
        "salary_band",
        "salary_range_usd"
    ]
].head(20)

,annual_salary_usd,salary_band,salary_range_usd
0,239000.0,Very High,135000
1,166000.0,Very High,110000
2,360000.0,Very High,140000
3,161000.0,Very High,90000
4,283000.0,Very High,120000
5,155000.0,Very High,135000
6,197000.0,Very High,120000
7,221000.0,Very High,132000
8,145000.0,High,145000
9,90000.0,Medium,110000


In [29]:
# Skills 
feature_df[
    [
        "requires_python",
        "requires_sql",
        "requires_machine_learning",
        "requires_deep_learning",
        "requires_generative_ai"
    ]
].head()

,requires_python,requires_sql,requires_machine_learning,requires_deep_learning,requires_generative_ai
0,True,True,False,False,False
1,True,False,False,False,False
2,True,False,False,False,False
3,True,True,False,False,False
4,False,False,False,False,False


In [30]:
# Save
feature_df.to_csv(
    "../data/processed/ai_jobs_feature_engineered.csv",
    index=False
)

In [31]:
import os

os.path.exists(
    "../data/processed/ai_jobs_feature_engineered.csv"
)

True